# HELM — toxicité et quatre profils
Test séparé sur deux commentaires français écrits pour cette démonstration : un commentaire injurieux et un désaccord courtois. Modèle réel XLM-R fine-tuné toxicité (révision figée dans HELM), aucun modèle de sentiment utilisé comme substitut. Ce test fonctionnel ne mesure pas les performances sur un corpus représentatif.

## 1. Télécharger les sources GitHub
Révision 612b87c6856d3d41ea069b0243d162f033538e55. Dépôt privé : lien d’archive temporaire saisi masqué, à renouveler après expiration.


**Version colorée 0.2.0a3** : quatre profils accessibles par onglets HTML autonomes ; les contributions restent signées et les méthodes distinctes.


In [ ]:
import hashlib
import io
import sys
import urllib.request
import zipfile
from pathlib import Path
from getpass import getpass

REVISION = "612b87c6856d3d41ea069b0243d162f033538e55"
if globals().get("ARCHIVE_REVISION") != REVISION:
    archive_url = getpass("Lien GitHub temporaire de la révision : ")
    assert archive_url.startswith("https://codeload.github.com/KossiFO/helm-xai-these/")
    archive = urllib.request.urlopen(archive_url, timeout=120).read()
    del archive_url
    ARCHIVE_REVISION = REVISION
source_dir = Path("/content/helm_github_source")
source_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
    root_name = zipped.namelist()[0].split("/")[0]
    archive_revision = root_name.rsplit("-", 1)[-1]
    assert len(archive_revision) >= 7 and REVISION.startswith(archive_revision), "Révision inattendue"
    assert all((source_dir / n).resolve().is_relative_to(source_dir.resolve()) for n in zipped.namelist())
    zipped.extractall(source_dir)
SOURCE = source_dir / root_name
for module_name in list(sys.modules):
    if module_name == "helm_colab" or module_name.startswith("helm_colab."):
        del sys.modules[module_name]
sys.path.insert(0, str(SOURCE / "examples/colab"))
print("Codes récupérés depuis GitHub :", REVISION)
print("Archive SHA256 :", hashlib.sha256(archive).hexdigest())
print("Sources consultables dans le panneau Fichiers :", SOURCE)


## 2. Installer HELM dans Colab
Environnement Python 3.12 séparé, CPU ; le premier téléchargement du modèle demande quelques minutes.


In [ ]:
from helm_colab.toxicity_session import ToxicityExperiment
toxicite = ToxicityExperiment()
toxicite.install()


## 3. Calculer pour les quatre profils
HELM choisit parmi SHAP, LIME et les contrefactuels selon sa matrice et les contraintes du profil. Les autres méthodes sont exclues explicitement avant le budget. Graine 42, 8 mots, évaluation k=3. Le même modèle et les mêmes commentaires sont utilisés pour chaque profil. Aucun apprentissage de préférence ni retour utilisateur simulé.


In [ ]:
toxicite.run()


## 4. Explorer les explications
Choisir le commentaire puis le profil. Le profil utilisateur explique le classement ; le modérateur reçoit les éléments à examiner ; l’expert compare les attributions ; l’audit donne la provenance.


In [ ]:
toxicite.show()


## 5. Code et audit
Les scripts sont consultables dans le dossier téléchargé depuis GitHub.


In [ ]:
from IPython.display import Code, display
display(Code((SOURCE / "examples/colab/helm_colab/toxicity.py").read_text(), language="python"))
toxicite.audit()
